

## 0. импорты + утилиты

сначала тащим библиотеки и функции для повторных прогонов (seed, графики ELBO, MCMC-диагностики).

In [ ]:
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, MCMC, NUTS
from pyro.infer.autoguide import (
    AutoDiagonalNormal,
    AutoMultivariateNormal,
    init_to_median,
    init_to_value,
)

from sklearn.cluster import KMeans
import arviz as az

In [ ]:

def fix_seed(seed: int) -> None:
    pyro.set_rng_seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)


def run_svi_once(
    *,
    model,
    guide_cls,
    init_loc_fn,
    seed: int,
    data,
    n_steps: int = 1800,
    lr: float = 0.01,
    label: str = "run",
):
    fix_seed(seed)
    pyro.clear_param_store()
    guide = guide_cls(model, init_loc_fn=init_loc_fn)
    svi = SVI(model, guide, pyro.optim.Adam({"lr": lr}), loss=Trace_ELBO())
    losses = []
    t0 = time.time()
    for _ in range(n_steps):
        losses.append(svi.step(data))
    dt = time.time() - t0
    with torch.no_grad():
        med = guide.median()
    print(f"[{label}] seed={seed} | {n_steps} steps | {dt:.1f}s | loss={losses[-1]:.2f}")
    return {"seed": seed, "time": dt, "losses": losses, "median": med}


def plot_loss_curves(runs, title: str):
    plt.figure(figsize=(9, 4))
    for r in runs:
        plt.plot(r["losses"], label=f"seed={r['seed']} ({r['time']:.1f}s)")
    plt.title(title)
    plt.xlabel("step")
    plt.ylabel("ELBO")
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.show()


def plot_diag_ellipses(data, med, title: str):
    locs = med["locs"].cpu().numpy()
    scales = med["scales"].cpu().numpy()
    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.3, c="gray")
    ax = plt.gca()
    palette = ["#2ecc71", "#9b59b6", "#e74c3c"]
    for i in range(locs.shape[0]):
        ax.add_patch(
            Ellipse(
                xy=locs[i],
                width=4 * scales[i, 0],
                height=4 * scales[i, 1],
                angle=0,
                edgecolor=palette[i % 3],
                fill=False,
                linewidth=2.5,
                label=f"c{i}",
            )
        )
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()


def plot_full_ellipses(data, med, title: str):
    locs = med["locs"].cpu().numpy()
    scales = med["scales"].cpu().numpy()
    corr_cholesky = med["corr_cholesky"].cpu().numpy()
    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.3, c="gray")
    ax = plt.gca()
    palette = ["#2ecc71", "#9b59b6", "#e74c3c"]
    for i in range(locs.shape[0]):
        L = np.diag(scales[i]) @ corr_cholesky[i]
        cov = L @ L.T
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        w, h = 4 * np.sqrt(vals)
        ax.add_patch(
            Ellipse(
                xy=locs[i], width=w, height=h, angle=theta,
                edgecolor=palette[i % 3], fill=False, linewidth=2.5, label=f"c{i}",
            )
        )
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()


def kmeans_init_fn(data, K: int):
    km = KMeans(n_clusters=K, n_init=10, random_state=17).fit(data.detach().cpu().numpy())
    centers = torch.tensor(km.cluster_centers_, dtype=torch.float)

    def init_loc(site):
        if site["name"] == "locs":
            return centers
        if site["name"] == "scales":
            shape = site["fn"].batch_shape + site["fn"].event_shape
            return torch.ones(shape, device=centers.device)
        return init_to_median(site)

    return init_loc


def run_mcmc_diag(*, model, data, warmup: int, num_samples: int = 300, num_chains: int = 2, label: str):
    fix_seed(0)
    cpu_data = data.detach().cpu() if isinstance(data, torch.Tensor) else data
    nuts = NUTS(model, init_strategy=init_to_median)
    mcmc = MCMC(nuts, num_samples=num_samples, warmup_steps=warmup, num_chains=num_chains)
    t0 = time.time()
    mcmc.run(cpu_data)
    dt = time.time() - t0
    print(f"[{label}] warmup={warmup} | {dt:.1f}s")
    az_data = az.from_pyro(mcmc)
    print(az.summary(az_data, var_names=["weights", "locs"], round_to=2))
    az.plot_trace(az_data, var_names=["weights", "locs"])
    az.plot_autocorr(az_data, var_names=["locs"], combined=True)
    plot_mcmc_results(obs_xy.numpy(), mcmc, f"{label} (warmup={warmup})")
    return mcmc, dt


def run_sparse_nuts(*, warmup: int = 100, num_samples: int = 300, num_chains: int = 2):
    nuts = NUTS(gmm_sparse, init_strategy=init_to_median)
    mcmc = MCMC(nuts, num_samples=num_samples, warmup_steps=warmup, num_chains=num_chains)
    t0 = time.time()
    mcmc.run(obs_xy)
    dt = time.time() - t0
    print(f"[sparse NUTS] warmup={warmup} | {dt:.1f}s")
    az_data = az.from_pyro(mcmc)
    print(az.summary(az_data, var_names=["weights"], round_to=3))
    az.plot_trace(az_data, var_names=["weights"])
    return mcmc, dt


**Игорь**, генерю кривые 2D-данные (3 кластера, один вытянутый). seed=17, не как у меня в черновике на 42.

In [ ]:
def make_twisted_gmm(n_samples=1500):
    torch.manual_seed(17)

    # 1) Кластер A: широкий (n=600)
    loc_a = torch.tensor([7.5, 2.5])
    data_a = torch.randn(600, 2) * 1.8 + loc_a
    labels_a = torch.zeros(600)

    # 2) Кластер B: вытянутый по диагонали (n=600)
    loc_b = torch.tensor([5.0, 5.0])
    scale_b = torch.tensor([4.5, 0.6])
    theta = torch.tensor([torch.pi / 4])  # 45 градусов
    rot = torch.tensor([[theta.cos(), -theta.sin()], [theta.sin(), theta.cos()]])
    data_b = (torch.randn(600, 2) * scale_b) @ rot.T + loc_b
    labels_b = torch.ones(600)

    # 3) Кластер C: узкий пик рядом с B (n=300)
    loc_c = torch.tensor([3.5, 5.5])
    data_c = torch.randn(300, 2) * 0.2 + loc_c
    labels_c = torch.ones(300) * 2

    # Объединение
    data = torch.cat([data_a, data_b, data_c])
    labels = torch.cat([labels_a, labels_b, labels_c])
    # Перемешивание с сохранением соответствия меток
    idx = torch.randperm(n_samples)
    return data[idx], labels[idx]


obs_xy, true_labels = make_twisted_gmm()

plt.figure(figsize=(10, 7))
colors = ['#1f77b4', '#ff7f0e', '#d62728']
for i, lbl in enumerate(['A', 'B', 'C']):
    mask = (true_labels == i)
    plt.scatter(obs_xy[mask, 0], obs_xy[mask, 1], 
                s=10, alpha=0.6, c=colors[i], label=f'Cluster {lbl}')


plt.title("Ground Truth")
plt.legend()
#plt.axis('equal')
plt.xlim(-8, 15)
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
def gmm_diagonal(data, K=3):
    D = data.shape[1]  # Размерность данных
    # Dirichlet(1.0) равномерное распределение весов компонентов смеси
    weights = pyro.sample("weights", dist.Dirichlet(torch.ones(K)))
    with pyro.plate("components", K):
        # Средние для каждого кластера
        locs = pyro.sample("locs", dist.Normal(torch.zeros(D), 1.0).to_event(1))
        # Масштабы для каждого кластера
        scales = pyro.sample("scales", dist.LogNormal(torch.zeros(D), 1.0).to_event(1))

    mixing_dist = dist.Categorical(weights)
    # Объединение компонентов в одну смесь
    component_dist = dist.Normal(locs, scales).to_event(1)
    mixture = dist.MixtureSameFamily(mixing_dist, component_dist)
    
    with pyro.plate("data", len(data)):
        pyro.sample("obs", mixture, obs=data)


pyro.render_model(gmm_diagonal, model_args=(obs_xy,), render_distributions=True)

**Игорь**, тут я собираю guide для basic VI — `AutoDiagonalNormal`, старт через `init_to_median`.
типа градиентный шаг.

In [ ]:
from pyro.infer.autoguide import AutoDiagonalNormal, init_to_median

pyro.clear_param_store()

guide = AutoDiagonalNormal(gmm_diagonal, init_loc_fn=init_to_median)

**прогон basic VI** SVI крутим ~1800 шагов, loss пишем в список, время замеряем (чтоб потом хвастаться или плакать).

In [ ]:
adam = pyro.optim.Adam({"lr": 0.01})
svi = SVI(gmm_diagonal, guide, adam, loss=Trace_ELBO())

# Цикл обучения
n_steps = 1800
losses = []

start_time = time.time()
for step in range(n_steps):
    loss = svi.step(obs_xy)
    losses.append(loss)
    if step % 500 == 0:
        print(f"Step {step} - Loss: {loss:.2f}")

duration = time.time() - start_time
print(f"Обучение завершено за {duration:.2f} сек.")

глянем ELBO: если к концу не успокоился можно добавить шагов, но обычно хватает.
я насрал

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title("ELBO Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

достаём median из guide центры и scales, дальше рисуем эллипсы 2σ.

In [ ]:
with torch.no_grad():
    medians = guide.median()
    final_locs = medians['locs']
    final_scales = medians['scales']

рисую 2σ эллипсы для basic VI. диагональная ковариация = эллипс **не крутится**, запомни это.

In [ ]:
from matplotlib.patches import Ellipse

def plot_results(data, locs, scales, title):
    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.3, c='gray')
    
    ax = plt.gca()
    for i in range(len(locs)):
        # Эллипс 2-sigma (накрывает ~95% данных)
        # width и height диаметры, поэтому умножаю scale на 2 для sigma и на 2 жля диаметра = 4
        el = Ellipse(xy=locs[i], width=4*scales[i, 0], height=4*scales[i, 1], 
                     angle=0, color=colors[i], fill=False, linewidth=3, label=f'Cluster {i}')
        ax.add_patch(el)
        
    plt.title(title)
    plt.legend()
    plt.show()

plot_results(obs_xy, final_locs, final_scales, "Basic VI Results (Diagonal Covariance)")

### прогон: basic VI, разные seed (сразу тут, не в конце)

**Игорь**, я гоняю три seed подряд смотри, насколько ELBO прыгает. Это и есть «стабильность VI»

я насрал снова

In [ ]:

seeds = [0, 1, 2]
runs_basic = [
    run_svi_once(
        model=gmm_diagonal,
        guide_cls=AutoDiagonalNormal,
        init_loc_fn=init_to_median,
        seed=s,
        data=obs_xy,
        label="basic / median init",
    )
    for s in seeds
]
plot_loss_curves(runs_basic, "Basic VI — разные seed")
plot_diag_ellipses(obs_xy.numpy(), runs_basic[0]["median"], "Basic VI эллипсы (seed=0)")


**k-means init** эмпирический байес по-студенчески: sklearn нашёл центры, мы их отдали в `locs`.
это не «чистый байес», но преподам обычно ок, если честно написать.

In [ ]:
km = KMeans(n_clusters=3, n_init=10).fit(obs_xy.numpy())
km_centers = torch.tensor(km.cluster_centers_, dtype=torch.float)

def init_loc_fn(site):
    if site["name"] == "locs":
        return km_centers
    if site["name"] == "scales":
        shape = site["fn"].batch_shape + site["fn"].event_shape
        return torch.ones(shape, device=km_centers.device)
    return init_to_median(site)

pyro.clear_param_store()
guide_km = AutoDiagonalNormal(gmm_diagonal, init_loc_fn=init_loc_fn)

ещё раз SVI, но guide уже с k-means стартом сравни с прошлым прогоном.

In [ ]:
adam = pyro.optim.Adam({"lr": 0.01})
svi = SVI(gmm_diagonal, guide_km, adam, loss=Trace_ELBO())

n_steps = 1800
losses = []
start_time = time.time()

for step in range(n_steps):
    loss = svi.step(obs_xy)
    losses.append(loss)
    if step % 500 == 0:
        print(f"Step {step} - Loss: {loss:.2f}")

duration = time.time() - start_time
print(f"Обучение завершено за {duration:.2f} сек.")

визуал basic VI после k-means init.

In [ ]:
import torch
from matplotlib.patches import Ellipse

with torch.no_grad():
    medians = guide_km.median()
    final_locs = medians['locs']
    final_scales = medians['scales']

plt.figure(figsize=(10, 7))
plt.scatter(obs_xy[:, 0], obs_xy[:, 1], s=10, alpha=0.3, c='gray')
ax = plt.gca()

colors = ['#1f77b4', '#ff7f0e', '#d62728']
for i in range(len(final_locs)):
    el = Ellipse(xy=final_locs[i], width=4*final_scales[i, 0], height=4*final_scales[i, 1], 
                 angle=0, color=colors[i], fill=False, linewidth=3, label=f'Cluster {i}')
    ax.add_patch(el)

plt.title("Basic VI with K-means Initialization")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

### прогон: basic VI + k-means init, три seed сравни с прошлым блоком

In [ ]:

init_km = kmeans_init_fn(obs_xy, K=3)
runs_basic_km = [
    run_svi_once(
        model=gmm_diagonal,
        guide_cls=AutoDiagonalNormal,
        init_loc_fn=init_km,
        seed=s,
        data=obs_xy,
        label="basic / kmeans init",
    )
    for s in seeds
]
plot_loss_curves(runs_basic_km, "Basic VI + k-means init")
plot_diag_ellipses(obs_xy.numpy(), runs_basic_km[0]["median"], "Basic+kmeans (seed=0)")


**full-rank модель** LKJ + полная ковариация.
вытянутый кластер B наконец можно описать нормальным повёрнутым эллипсом, не квадратом из basic VI.

In [ ]:
import pyro.distributions as dist

def gmm_full_rank(data, K=3):
    D = data.shape[1]
    weights = pyro.sample("weights", dist.Dirichlet(torch.ones(K)))
    
    with pyro.plate("components", K):
        # Средние
        locs = pyro.sample("locs", dist.Normal(torch.zeros(D), 10.).to_event(1))
        # Работа с матрицей ковариации через разложение Холецкого
        scales = pyro.sample("scales", dist.HalfNormal(torch.ones(D)).to_event(1))
        corr_cholesky = pyro.sample("corr_cholesky", dist.LKJCholesky(D, eta=1.0))
        L_omega = torch.diag_embed(scales) @ corr_cholesky
    # Смесь с MultivariateNormal
    mixing_dist = dist.Categorical(weights)
    component_dist = dist.MultivariateNormal(locs, scale_tril=L_omega)
    mixture = dist.MixtureSameFamily(mixing_dist, component_dist)
    
    with pyro.plate("data", len(data)):
        pyro.sample("obs", mixture, obs=data)

# Full Rank гайд
from pyro.infer.autoguide import AutoMultivariateNormal

pyro.clear_param_store()
# Та же K-means инициализацию, что и раньше
guide_full = AutoMultivariateNormal(gmm_full_rank, init_loc_fn=init_to_median)

full rank VI: `AutoMultivariateNormal`, SVI, loss, эллипсы с углом.

In [ ]:
import torch
import pyro
import pyro.distributions as dist
import time
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoMultivariateNormal
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np

def gmm_full_rank(data, K=3):
    D = data.shape[1]
    weights = pyro.sample("weights", dist.Dirichlet(torch.ones(K)))
    
    with pyro.plate("components", K):
        locs = pyro.sample("locs", dist.Normal(torch.zeros(D), 10.0).to_event(1))
        scales = pyro.sample("scales", dist.HalfNormal(torch.ones(D)).to_event(1))
        corr_cholesky = pyro.sample("corr_cholesky", dist.LKJCholesky(D, concentration=torch.tensor(1.0)))
        L_omega = torch.diag_embed(scales) @ corr_cholesky

    mixing_dist = dist.Categorical(weights)
    component_dist = dist.MultivariateNormal(locs, scale_tril=L_omega)
    mixture = dist.MixtureSameFamily(mixing_dist, component_dist)
    
    with pyro.plate("data", len(data)):
        pyro.sample("obs", mixture, obs=data)

# Подготовка обучения
pyro.clear_param_store()
guide_full = AutoMultivariateNormal(gmm_full_rank, init_loc_fn=init_to_median)

adam = pyro.optim.Adam({"lr": 0.01})
svi = SVI(gmm_full_rank, guide_full, adam, loss=Trace_ELBO())

# Циклы обучения
n_steps = 1800
losses_full = []

print("Начинаем обучение Full Rank VI...")
start_time = time.time()

for step in range(n_steps):
    loss = svi.step(obs_xy)
    losses_full.append(loss)
    if step % 500 == 0:
        print(f"Step {step} - Loss: {loss:.2f}")

duration_full = time.time() - start_time
print(f"Обучение завершено за {duration_full:.2f} сек.")

# Визуализация результата
def plot_full_rank_results(data, guide):
    with torch.no_grad():
        medians = guide.median()
        locs = medians['locs'].numpy()
        scales = medians['scales'].numpy()
        corr_cholesky = medians['corr_cholesky'].numpy()
        
    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.3, c='gray')
    ax = plt.gca()
    colors = ['#1f77b4', '#ff7f0e', '#d62728']
    
    for i in range(len(locs)):
        # Матрица ковариации Sigma = L 
        L = np.diag(scales[i]) @ corr_cholesky[i]
        cov = L @ L.T
        
        # Форма эллипса
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        width, height = 4 * np.sqrt(vals) 
        
        el = Ellipse(xy=locs[i], width=width, height=height, angle=theta,
                     edgecolor=colors[i], fill=False, linewidth=3, label=f'Cluster {i}')
        ax.add_patch(el)

    plt.title(f"Full Rank VI Results (Time: {duration_full:.2f}s)")
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

plot_full_rank_results(obs_xy, guide_full)

### прогон: full-rank VI, три seed

тут уже эллипсы **крутятся** для кластера B это ближе к правде, чем basic VI.

In [ ]:

runs_full = [
    run_svi_once(
        model=gmm_full_rank,
        guide_cls=AutoMultivariateNormal,
        init_loc_fn=init_to_median,
        seed=s,
        data=obs_xy,
        label="full-rank / median",
    )
    for s in seeds
]
plot_loss_curves(runs_full, "Full-rank VI — разные seed")
plot_full_ellipses(obs_xy.numpy(), runs_full[0]["median"], "Full-rank (seed=0)")

init_km_full = kmeans_init_fn(obs_xy, K=3)
runs_full_km = [
    run_svi_once(
        model=gmm_full_rank,
        guide_cls=AutoMultivariateNormal,
        init_loc_fn=init_km_full,
        seed=s,
        data=obs_xy,
        label="full-rank / kmeans",
    )
    for s in seeds
]
plot_loss_curves(runs_full_km, "Full-rank VI + k-means init")
plot_full_ellipses(obs_xy.numpy(), runs_full_km[0]["median"], "Full-rank+kmeans (seed=0)")


модель под MCMC/NUTS та же идея full-rank, но уже сэмплим, не оптимизируем.

In [ ]:
from pyro.infer import MCMC, NUTS

def gmm_for_mcmc(data, K=3):
    D = data.shape[1]
    weights = pyro.sample("weights", dist.Dirichlet(torch.ones(K)))
    
    with pyro.plate("components", K):
        locs = pyro.sample("locs", dist.Normal(torch.zeros(D), 10.0).to_event(1))
        scales = pyro.sample("scales", dist.HalfNormal(torch.ones(D)).to_event(1))
        corr_cholesky = pyro.sample("corr_cholesky", dist.LKJCholesky(D, concentration=1.0))
        L_omega = torch.diag_embed(scales) @ corr_cholesky

    component_dist = dist.MultivariateNormal(locs, scale_tril=L_omega)
    mixture = dist.MixtureSameFamily(dist.Categorical(weights), component_dist)
    
    with pyro.plate("data", len(data)):
        pyro.sample("obs", mixture, obs=data)

**прогон NUTS** с warmup 20 и 100 сразу смотрим время (MCMC не для слабонервных).

In [ ]:
def run_mcmc(warmup, num_samples=300):
    pyro.clear_param_store()
    nuts_kernel = NUTS(gmm_for_mcmc)
    mcmc = MCMC(nuts_kernel, num_samples=num_samples, warmup_steps=warmup)
    
    start_time = time.time()
    mcmc.run(obs_xy)
    duration = time.time() - start_time
    
    print(f"MCMC ({warmup} warmup) завершен за {duration:.2f} сек.")
    return mcmc, duration

mcmc_20, time_20 = run_mcmc(20)
mcmc_100, time_100 = run_mcmc(100)

ArviZ: trace, R-hat, autocorr если R-hat далёк от 1, это не jojo, это плохо.

In [ ]:
import arviz as az

# Конвертирую данные Pyro в формат ArviZ
data_az_20 = az.from_pyro(mcmc_20)
data_az_100 = az.from_pyro(mcmc_100)

az.plot_trace(data_az_100, var_names=["locs"])
plt.show()

print(az.summary(data_az_100, var_names=["locs", "weights"]))

az.plot_autocorr(data_az_100, var_names=["locs"], combined=True)
plt.show()

средние по сэмплам MCMC чтоб потом нарисовать «типа постериор».

In [ ]:
samples = mcmc_100.get_samples()
mean_locs = samples['locs'].mean(0)
mean_scales = samples['scales'].mean(0)
mean_corr = samples['corr_cholesky'].mean(0)

функция рисования MCMC-эллипсов (уже с поворотом из `corr_cholesky`).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_mcmc_results(data, mcmc_obj, title="MCMC Results"):
    samples = mcmc_obj.get_samples()
    
    with torch.no_grad():
        locs = samples['locs'].mean(0).numpy()        
        scales = samples['scales'].mean(0).numpy()   
        corr_cholesky = samples['corr_cholesky'].mean(0).numpy() 

    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.3, c='gray')
    ax = plt.gca()
    colors = ['#1f77b4', '#ff7f0e', '#d62728']
    
    for i in range(len(locs)):
        # Восстанавливаю матрицу ковариации из средних значений
        L = np.diag(scales[i]) @ corr_cholesky[i]
        cov = L @ L.T
        
        # Считаю геометрию эллипса
        vals, vecs = np.linalg.eigh(cov)
        order = vals.argsort()[::-1]
        vals, vecs = vals[order], vecs[:, order]
        theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        width, height = 4 * np.sqrt(vals) # 2-sigma
        
        el = Ellipse(xy=locs[i], width=width, height=height, angle=theta,
                     edgecolor=colors[i], fill=False, linewidth=3, label=f'Cluster {i}')
        ax.add_patch(el)

    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.2)
    plt.show()

### NUTS на full-rank (warmup 20 vs 100) + ArviZ

**Игорь**, это у меня раньше висело **в конце** ноутбука пачкой здесь сразу после MCMC-модели.

In [ ]:

mcmc_w20, _ = run_mcmc_diag(model=gmm_full_rank, data=obs_xy, warmup=20, label="NUTS full")
mcmc_w100, _ = run_mcmc_diag(model=gmm_full_rank, data=obs_xy, warmup=100, label="NUTS full")


пробую явный init из k-means для NUTS посмотрим, ускорит ли сходимость.

In [ ]:
init_params = {
    "locs": km_centers, 
    "weights": torch.ones(3) / 3, 
    "scales": torch.ones(3, 2),    
    "corr_cholesky": torch.eye(2).repeat(3, 1, 1) 
}

from pyro.infer.autoguide import init_to_value

nuts_kernel = NUTS(gmm_full_rank, init_strategy=init_to_value(values=init_params))
mcmc_100 = MCMC(nuts_kernel, num_samples=300, warmup_steps=100)

print("Запуск MCMC с явной инициализацией...")
mcmc_100.run(obs_xy)

plot_mcmc_results(obs_xy, mcmc_100, "MCMC (Warmup: 100) with K-means Init")

**sparse GMM**, K=10, Dirichlet(0.1) лишние компоненты сами заглохнут.
как с десятью руками, но работают только три.

In [ ]:
def gmm_sparse(data, K=10):
    D = data.shape[1]
    weights = pyro.sample("weights", dist.Dirichlet(torch.ones(K) * 0.1))
    
    with pyro.plate("components", K):
        locs = pyro.sample("locs", dist.Normal(torch.zeros(D), 10.).to_event(1))
        scales = pyro.sample("scales", dist.HalfNormal(torch.ones(D)).to_event(1))
        corr_cholesky = pyro.sample("corr_cholesky", dist.LKJCholesky(D, concentration=1.0))
        L_omega = torch.diag_embed(scales) @ corr_cholesky

    mixture = dist.MixtureSameFamily(dist.Categorical(weights), dist.MultivariateNormal(locs, scale_tril=L_omega))
    with pyro.plate("data", len(data)):
        pyro.sample("obs", mixture, obs=data)

pyro.clear_param_store()
guide_sparse = AutoMultivariateNormal(gmm_sparse, init_loc_fn=init_to_median)
svi = SVI(gmm_sparse, guide_sparse, pyro.optim.Adam({"lr": 0.01}), loss=Trace_ELBO())

for i in range(2000):
    svi.step(obs_xy)

рисую sparse: красным компоненты с весом > 0.05, остальные серые призраки.

In [ ]:
def plot_sparse_results(data, guide):
    with torch.no_grad():
        medians = guide.median()
        locs = medians['locs'].numpy()
        scales = medians['scales'].numpy()
        corr_cholesky = medians['corr_cholesky'].numpy()
        weights = medians['weights'].numpy()
        
    plt.figure(figsize=(10, 7))
    plt.scatter(data[:, 0], data[:, 1], s=10, alpha=0.2, c='gray')
    ax = plt.gca()
    
    for i in range(len(weights)):
        L = np.diag(scales[i]) @ corr_cholesky[i]
        cov = L @ L.T
        vals, vecs = np.linalg.eigh(cov)
        theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
        width, height = 4 * np.sqrt(vals)
        
        color = 'red' if weights[i] > 0.05 else 'lightgray'
        alpha = 0.8 if weights[i] > 0.05 else 0.3
        
        el = Ellipse(xy=locs[i], width=width, height=height, angle=theta,
                     edgecolor=color, fill=False, linewidth=2, alpha=alpha)
        ax.add_patch(el)
    plt.title(f"Sparse Dirichlet Clustering (K=10, alpha=0.1). Weights: {np.round(weights, 2)}")
    plt.show()

plot_sparse_results(obs_xy, guide_sparse)

### sparse GMM + NUTS (K=10) тоже тут, не в хвосте. MCMC на 10 компонентах = долго, заранее сори.

In [ ]:
run_sparse_nuts(warmup=100)

---

## итог для тебя(коротко)

- **basic VI** быстро, но диагональ не крутит эллипсы на вытянутом кластере мимо.
- **full-rank VI** тяжелее, зато форма норм.
- **k-means init** читерский, но рабочий старт; в отчёте напиши честно.
- **MCMC** медленный, но «честнее»; смотри R-hat / trace.
- **sparse Dirichlet(0.1)** сам вырубает лишние компоненты.



*ゴ・ゴ・ゴ・ゴ* = shift+enter